## Ayudantía 2
#### Regresión Lineal

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split    
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score # MSE y R2
from sklearn.preprocessing import StandardScaler, LabelEncoder
from prettytable import PrettyTable

In [20]:
df["description"].iloc[0] # bag of words, TF-IDF, word embeddings
type(df["priceSqFt"].iloc[0]) 
df["numBalconies"].value_counts()

numBalconies
2.0    2187
1.0     376
3.0     110
4.0      57
5.0       3
6.0       3
8.0       1
Name: count, dtype: int64

In [19]:
df = pd.read_csv('housing_Tarea2.csv')
df

,house_type,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,verificationDate,description,SecurityDeposit,Status
0,1 RK Studio Apartment,400 sq ft,Kalkaji,Delhi,28.545561,77.254349,22000,INR,1.0,NaN,NaN,NaN,Posted a day ago,"Fully furnished, loaded with amenities & gadge...",No Deposit,Furnished
1,1 RK Studio Apartment,400 sq ft,Mansarover Garden,Delhi,28.643259,77.132828,20000,INR,1.0,NaN,NaN,NaN,Posted 9 days ago,Here is an excellent 1 BHK Independent Floor a...,No Deposit,Furnished
2,2 BHK Independent Floor,500 sq ft,Uttam Nagar,Delhi,28.618677,77.053352,8500,INR,1.0,NaN,NaN,NaN,Posted 12 days ago,"Zero Brokerage.\n\n2 Room set, Govt bijali Met...",No Deposit,Semi-Furnished
3,3 BHK Independent House,"1,020 sq ft",Model Town,Delhi,28.712898,77.180000,48000,INR,3.0,NaN,NaN,NaN,Posted a year ago,Itâs a 3 bhk independent house situated in M...,No Deposit,Furnished
4,2 BHK Apartment,810 sq ft,Sector 13 Rohini,Delhi,28.723539,77.131424,20000,INR,2.0,NaN,NaN,NaN,Posted a year ago,Well designed 2 bhk multistorey apartment is a...,No Deposit,Unfurnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4 BHK Villa,"5,896 sq ft",Sunder Nagar,Delhi,28.618437,76.961784,1022001,INR,4.0,2.0,NaN,NaN,Posted 2 months ago,Its four bhk villa in the super location of De...,"40,10,102",Unfurnished
4996,5 BHK Independent House,"6,521 sq ft",Sunder Nagar,Delhi,28.618437,76.961784,1549181,INR,4.0,2.0,NaN,NaN,Posted 2 months ago,A 5 bhk property is available for rent in Sund...,"54,01,015",Unfurnished
4997,3 BHK Independent Floor,"1,855 sq ft",New Friends Colony,Delhi,28.567051,77.273560,301012,INR,3.0,2.0,NaN,NaN,Posted 2 months ago,Its three bhk builder floor in the super locat...,"18,18,181",Unfurnished
4998,3 BHK Independent Floor,"2,856 sq ft",New Friends Colony,Delhi,28.567051,77.273560,301011,INR,3.0,2.0,NaN,NaN,Posted 2 months ago,Its three bhk builder floor in the super locat...,"10,10,110",Unfurnished


## Cosas que no sirven para ingresarlos al modelo
#### NaN = Not a Number (tipo de dato ""void"")
#### cadenas de texto
#### EDA = Exploratory Data Analysis

# Limpieza de Datos

In [20]:
# Limpieza de datos para la columna house_type
# "mujer" "hombre" <- 0 1 <- encoding, con el tipo de Label
df["numRooms"] = df["house_type"].str[0].astype(int)
# 3 BHK <- 3 Bedroom, 1 Hall, 1 Kitchen
df["house_type"].value_counts()
df["rooms"] = df["house_type"].str.split(" ").str[1]
df["housing_type"] = df["house_type"].str.split(" ").str[2:].str.join(" ")

# Encoding características categóricas
encoder = LabelEncoder()
df["rooms"] = encoder.fit_transform(df["rooms"]).astype(int)
df["housing_type"] = encoder.fit_transform(df["housing_type"]).astype(int)
df.drop(columns=["house_type"], inplace=True)

In [21]:
# Limpieza columna house_size
df["house_size"] = df["house_size"].str.replace(",","") 
df["house_size"] = df["house_size"].str.split(" ").str[0].astype(int)

In [22]:
df["location"] = encoder.fit_transform(df["location"]).astype(int)
df["city"] = encoder.fit_transform(df["city"]).astype(int)
df["currency"] = encoder.fit_transform(df["currency"]).astype(int)
df = df.fillna(0) # Rellenar valores nulos con 0


In [23]:
verif_map = {
    'day' : 1, 'days' : 1,
    "week" : 7, "weeks" : 7, 
    "month" : 30, "months" : 30,
    "year" : 365, "year" : 365
}
df[["cant_veces_str", "mult"]] = df["verificationDate"].str.extract(r'Posted\s+(\d+|a|an)\s+(\w+)')
df['cant_veces'] = df['cant_veces_str'].replace({'a': 1, 'an': 1}).astype(int)
df["days_since_verif"] = df["cant_veces"] * df["mult"].map(verif_map) # .apply()
df.drop(columns=["verificationDate", "cant_veces_str", "mult", "cant_veces"], inplace=True)
df.drop(columns=["description" ], inplace=True)
df

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,isNegotiable,priceSqFt,SecurityDeposit,Status,numRooms,rooms,housing_type,days_since_verif
0,400,88,0,28.545561,77.254349,22000,0,1.0,0.0,0,0.0,No Deposit,Furnished,1,1,3,1.0
1,400,124,0,28.643259,77.132828,20000,0,1.0,0.0,0,0.0,No Deposit,Furnished,1,1,3,9.0
2,500,259,0,28.618677,77.053352,8500,0,1.0,0.0,0,0.0,No Deposit,Semi-Furnished,2,0,1,12.0
3,1020,133,0,28.712898,77.180000,48000,0,3.0,0.0,0,0.0,No Deposit,Furnished,3,0,2,365.0
4,810,201,0,28.723539,77.131424,20000,0,2.0,0.0,0,0.0,No Deposit,Unfurnished,2,0,0,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,5896,249,0,28.618437,76.961784,1022001,0,4.0,2.0,0,0.0,"40,10,102",Unfurnished,4,0,4,60.0
4996,6521,249,0,28.618437,76.961784,1549181,0,4.0,2.0,0,0.0,"54,01,015",Unfurnished,5,0,2,60.0
4997,1855,146,0,28.567051,77.273560,301012,0,3.0,2.0,0,0.0,"18,18,181",Unfurnished,3,0,1,60.0
4998,2856,146,0,28.567051,77.273560,301011,0,3.0,2.0,0,0.0,"10,10,110",Unfurnished,3,0,1,60.0


In [124]:
df.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,4315.000000
mean,2982.885400,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,147.619235
std,2168.663368,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.706730
min,150.000000,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,1.000000
25%,1100.000000,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,20.000000
50%,2500.000000,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,150.000000
75%,5896.000000,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,14521.000000,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [126]:
df.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5.000000e+03,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,4315.000000
mean,-9.094947e-17,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,147.619235
std,1.000100e+00,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.706730
min,-1.306412e+00,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,1.000000
25%,-8.683108e-01,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,20.000000
50%,-2.226873e-01,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,150.000000
75%,1.343411e+00,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,5.320913e+00,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [125]:
scaler = StandardScaler()
df["house_size"] = scaler.fit_transform(df[["house_size"]])


In [24]:
df_baseline = df.copy()
df_baseline.describe()

,house_size,location,city,latitude,longitude,price,currency,numBathrooms,numBalconies,priceSqFt,numRooms,rooms,housing_type,days_since_verif
count,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5.000000e+03,5000.0,5000.000000,5000.000000,5000.0,5000.00000,5000.00000,5000.000000,4315.000000
mean,2982.885400,136.314400,0.0,28.578012,77.174499,2.221738e+05,0.0,2.904000,1.069800,0.0,3.09780,0.02760,1.210400,147.619235
std,2168.663368,79.323699,0.0,0.190186,0.115636,2.739843e+05,0.0,1.104458,1.053731,0.0,1.14162,0.16384,1.045347,136.706730
min,150.000000,0.000000,0.0,20.011379,72.771332,3.000000e+03,0.0,0.000000,0.000000,0.0,1.00000,0.00000,0.000000,1.000000
25%,1100.000000,65.000000,0.0,28.544489,77.138248,2.950000e+04,0.0,2.000000,0.000000,0.0,2.00000,0.00000,1.000000,20.000000
50%,2500.000000,141.000000,0.0,28.569295,77.196472,1.250000e+05,0.0,3.000000,1.000000,0.0,3.00000,0.00000,1.000000,150.000000
75%,5896.000000,195.000000,0.0,28.618687,77.228950,3.011020e+05,0.0,4.000000,2.000000,0.0,4.00000,0.00000,1.000000,180.000000
max,14521.000000,287.000000,0.0,28.805466,80.361313,3.010101e+06,0.0,10.000000,8.000000,0.0,9.00000,1.00000,5.000000,365.000000


In [25]:
df_baseline.drop(columns=["isNegotiable", "Status", "SecurityDeposit"], inplace=True)
df_baseline.fillna(0, inplace=True)

In [26]:
# Variables
X = df_baseline.drop(columns=["price"])  # Variables independientes
y = df_baseline["price"]                 # Variable dependiente

# División del conjunto
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=254)
 

In [ ]:
# Ajuste/Entrenamiento del modelo
lr = LinearRegression()
lr.fit(X_train, y_train)
y_prediction = lr.predict(X_test) # y_gorrito

In [28]:
mse = mean_squared_error(y_test, y_prediction)
r2 = r2_score(y_test, y_prediction)

In [31]:
table = PrettyTable()
table.field_names = ["Modelo", "MSE", "R2"]
table.add_row(["Baseline", mse, r2])
table.add_row(["Limpio", 0, 0])
table.add_row(["Limpio + interacción", 0, 0])
table

Modelo,MSE,R2
Baseline,36960214057.05083,0.56651655379927
Limpio,0,0
Limpio + interacción,0,0
